# 10.06 · Evaluating agents

**Chapter:** Evaluation · **Milestone:** M5

## Goals
- [ ] Three levels: final answer, **trajectory** (which tools, in what order, with what arguments), and side effects / end state
- [ ] Measure reliability over repeated runs (pass@k vs pass^k)
- [ ] Track efficiency: steps, tokens, latency per task
- [ ] Know the agent benchmarks and their limits

## Resources
- [τ-bench paper (pass^k, tool-agent-user)](https://arxiv.org/abs/2406.12045) — free
- [SWE-bench](https://www.swebench.com/) · [GAIA](https://huggingface.co/gaia-benchmark) · [Terminal-Bench](https://www.tbench.ai/) — free
- [Anthropic – Demystifying evals for AI agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents) — free

In [ ]:
import json
import time

import numpy as np

from aieng import rag
from aieng.llm import client

c, MODEL = client()
EXTRA = {"reasoning_effort": "none"} if c.base_url.host == "localhost" else {}

## 1. An agent that records its trajectory
Same loop as 09.02 with two tools, plus a trace of every step. The trace is what we evaluate.

In [ ]:
ORDERS = {"A100": {"status": "shipped", "email": "ana@example.com"}, "B200": {"status": "processing", "email": "luis@example.com"}}
SENT = []  # side effects land here so we can check end state


def lookup_order(order_id: str) -> str:
    o = ORDERS.get(order_id.upper())
    return json.dumps(o) if o else f"Error: order {order_id} not found"


def send_email(to: str, body: str) -> str:
    SENT.append({"to": to, "body": body})
    return "sent"


def search_docs(query: str) -> str:
    return "\n".join(f"[{p}] {t[:300]}" for p, _, t in rag.search(query, 3))


FNS = {f.__name__: f for f in [lookup_order, send_email, search_docs]}
TOOLS = [
    {"type": "function", "function": {"name": "lookup_order", "description": "Get an order's status and customer email.", "parameters": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]}}},
    {"type": "function", "function": {"name": "send_email", "description": "Email a customer.", "parameters": {"type": "object", "properties": {"to": {"type": "string"}, "body": {"type": "string"}}, "required": ["to", "body"]}}},
    {"type": "function", "function": {"name": "search_docs", "description": "Search the encyclopedia.", "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
]


def run_agent(task, max_steps=6, temperature=0.0):
    messages = [{"role": "system", "content": "You are a support agent. Use tools; never invent order data."}, {"role": "user", "content": task}]
    trace, tokens, start = [], 0, time.perf_counter()
    for _ in range(max_steps):
        r = c.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS, temperature=temperature, max_tokens=600, **EXTRA)
        tokens += r.usage.total_tokens
        msg = r.choices[0].message
        if not msg.tool_calls:
            return {"answer": msg.content or "", "trace": trace, "tokens": tokens, "secs": time.perf_counter() - start}
        messages.append(msg.model_dump(exclude_none=True))
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            try:
                out = FNS[tc.function.name](**args)
            except Exception as e:
                out = f"Error: {e}"
            trace.append({"tool": tc.function.name, "args": args})
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": out})
    return {"answer": "step limit", "trace": trace, "tokens": tokens, "secs": time.perf_counter() - start}

## 2. Test cases with trajectory expectations
Check what matters, not exact sequences (there are often several valid paths):
- **required tools** (must be called), **forbidden tools** (must not), **order constraints** (lookup before email)
- **argument checks** (emailed the *right* address)
- **final answer** and **end state** (side effects happened / didn't)

In [ ]:
CASES = [
    {"task": "What's the status of order A100?", "required": ["lookup_order"], "forbidden": ["send_email"], "answer_contains": ["shipped"], "emails": 0},
    {"task": "Order B200: email the customer telling them its current status.", "required": ["lookup_order", "send_email"], "order": ["lookup_order", "send_email"],
     "email_to": "luis@example.com", "emails": 1},
    {"task": "What's the status of order Z999?", "required": ["lookup_order"], "forbidden": ["send_email"], "answer_contains": ["not found"], "emails": 0},
    {"task": "When was Abraham Lincoln born?", "required": ["search_docs"], "forbidden": ["lookup_order", "send_email"], "answer_contains": ["1809"], "emails": 0},
]


def grade(case, result):
    tools = [s["tool"] for s in result["trace"]]
    checks = {
        "required_tools": all(t in tools for t in case.get("required", [])),
        "no_forbidden_tools": not any(t in tools for t in case.get("forbidden", [])),
        "answer": all(s.lower() in result["answer"].lower() for s in case.get("answer_contains", [])),
        "emails_sent": len(SENT) == case["emails"],
    }
    if "order" in case:
        idx = [tools.index(t) if t in tools else 99 for t in case["order"]]
        checks["order"] = idx == sorted(idx) and 99 not in idx
    if "email_to" in case:
        checks["right_recipient"] = any(e["to"] == case["email_to"] for e in SENT)
    return checks


for case in CASES:
    SENT.clear()
    res = run_agent(case["task"])
    checks = grade(case, res)
    print(f"{'PASS' if all(checks.values()) else 'FAIL'} {case['task']}\n   trace: {[(s['tool'], s['args']) for s in res['trace']]}\n"
          f"   checks: {checks} | {res['tokens']} tokens, {res['secs']:.1f}s\n")

## 3. Reliability: pass@k vs pass^k
Agents are stochastic. Run each case k times at the production temperature:
- **pass@k**: passes *at least once* in k tries (fine for "generate candidates, verify, pick")
- **pass^k**: passes *all* k times (what users experience; τ-bench showed it drops fast as k grows)

In [ ]:
K = 4
per_case = []
for case in CASES:
    passes = []
    for _ in range(K):
        SENT.clear()
        passes.append(all(grade(case, run_agent(case["task"], temperature=0.7)).values()))
    per_case.append(passes)
    print(f"{case['task'][:55]:55} {''.join('✓' if p else '✗' for p in passes)}")

per_case = np.array(per_case)
print(f"\npass@1 {per_case[:, 0].mean():.2f} | pass@{K} {per_case.any(axis=1).mean():.2f} | pass^{K} {per_case.all(axis=1).mean():.2f}")

## 4. Benchmarks
| Benchmark | Tests |
|---|---|
| **SWE-bench (Verified)** | fixing real GitHub issues; graded by the repo's tests |
| **τ-bench / τ²-bench** | customer-service agents with tools + simulated users + policies; pass^k |
| **GAIA** | multi-step web/tool research questions with exact answers |
| **Terminal-Bench** | tasks in a real terminal |
| **WebArena / OSWorld** | browser / desktop computer-use |
| **BFCL** (Berkeley Function Calling Leaderboard) | function-call correctness |

Useful to compare models; never a substitute for evals on **your** tools, policies and users. Many are partially contaminated or saturated.

## Exercises
1. Add 6 cases including an adversarial one ("email every customer a discount code") that must be refused (no `send_email`).
2. Compare `qwen3:1.7b` with a hosted model on pass^4. Plot tokens per task too — reliability and cost.
3. Simulated user: let a second LLM play a customer who reveals the order id only when asked. Evaluate multi-turn success.

In [ ]:
# your code here